# Object Detection using TensorFlow
### Intermediate Level | Live Coding Session — Your Own CNN + a Pretrained Locator, in One Notebook

**Instructor note:** Demonstrated in **VS Code**, but runs the same in **Jupyter Notebook**. In **Google Colab**, the final dashboard cell automatically uses Colab's port-forwarding.

**The split we're using in this project:**
- **Classification (is it a cat or a dog?)** — done by a **CNN we build and train ourselves**, from real cat/dog photos.
- **Localization (where is the animal in the photo?)** — done by a **pretrained detector** (SSD MobileNet V2, trained on COCO). This part is not trained by us, because real cat/dog datasets do not come with bounding box labels — nobody has hand-drawn a box around the animal in every training photo. Training our own box regressor would need a dataset that simply is not available here.

So the pretrained model never gets to decide "cat or dog" — it only draws a rough box around where an animal-shaped thing is. Our own trained CNN then looks specifically at what's inside that box and makes the actual cat/dog call.

1. **Section 1 — Imports and setup**
2. **Section 2 — Visualization utilities**
3. **Section 3 — Data extraction and preprocessing** (real cat/dog photos from `cats_vs_dogs`)
4. **Section 4 — Building our own CNN**
5. **Section 5 — Training and evaluating our CNN**
6. **Section 6 — Loading a pretrained detector, used only to find the box**
7. **Section 7 — Combining both: detect the box, classify with our own CNN**
8. **Section 8 — The Dashboard**, running and displayed right here at the end

**Agenda (60 minutes)**
1. Quick theory: why we split classification and localization this way (5 min)
2. Imports, visualization utilities (7 min)
3. Extracting and preprocessing real cat/dog data (10 min)
4. Building the CNN (8 min)
5. Training and evaluating (8 min live, full run left for students after class)
6. Loading the pretrained locator (5 min)
7. Combining both, testing on real photos (10 min)
8. Building and launching the dashboard (5 min)
9. Wrap up (2 min)


## Quick Theory: Why Split Classification and Localization?

- **Classification** answers *"what is this?"* — it needs a labeled dataset where every image has a class (cat / dog). `cats_vs_dogs` gives us exactly that, so we can train our own model on it.
- **Localization** answers *"where is it in the image?"* — it needs a dataset where every image also has a hand-drawn box around the object. `cats_vs_dogs` does **not** include that, and drawing boxes on thousands of photos ourselves is not realistic for one class.
- So we use a **pretrained detector** (already trained on COCO, a dataset that does include boxes) purely as a "where roughly is an animal" finder, and let **our own trained CNN** make the actual cat/dog decision on just that region.
- This pattern — reuse a pretrained model for the part you cannot easily train yourself, train your own model for the part you can — is extremely common in real projects.


## Section 1 — Imports and Setup
---
### 🎤 Speaking Notes
- "Before we write any deep learning code, let's make sure our environment is ready. This checks what's already installed instead of blindly reinstalling everything, so re-running it later is instant."
- "Notice this checks package *metadata*, not by actually importing the library — importing TensorFlow just to check it exists would itself take several seconds. We'll do the real import next."
- "The next cell does the real imports: TensorFlow for our own CNN, TensorFlow Hub for the pretrained locator we bring in later, and Flask for the dashboard at the very end. We're also detecting Colab here, because the dashboard needs to display a little differently there."


In [ ]:
# Fast dependency check: reads each package's installed metadata instead of
# actually importing it - actually importing a library just to check "is it
# installed" is itself slow (TensorFlow alone can take many seconds to load).
# The real (unavoidable) import cost happens naturally in the next cell, when
# we actually use these libraries.
#
# setuptools needs special handling: as of setuptools 82 (Feb 2026), the
# pkg_resources module that tensorflow_hub still depends on was REMOVED from
# setuptools entirely. A plain "is setuptools installed" check would pass even
# with the broken version 82+, so we specifically try to import pkg_resources
# itself and pin an older setuptools if that fails.
from importlib import metadata  # lets us check a package's installed version without importing the package itself
import subprocess  # lets us run "pip install" as a command from within Python
import sys  # gives us sys.executable, the exact Python interpreter running this notebook

try:
    import pkg_resources  # noqa: F401 - only checking that this succeeds
    pkg_resources_ok = True  # import succeeded, so pkg_resources is usable
except ImportError:
    pkg_resources_ok = False  # import failed - setuptools is missing or too new (82+)

required_packages = [
    "requests",  # used for downloading files with proper browser-like headers
    "datasets",  # Hugging Face datasets library - reliable, CDN-backed dataset downloads
    "tensorflow",  # core deep learning framework
    "tensorflow-hub",  # loads the pretrained locator model
    "matplotlib",  # plotting images and training curves
    "numpy",  # array math
    "pillow",  # image loading and drawing (PIL)
    "flask",  # powers the dashboard at the end of the notebook
]

missing = []  # will collect the names of any packages not currently installed
for package_name in required_packages:  # check each required package one at a time
    try:
        metadata.version(package_name)  # raises if the package isn't installed
    except metadata.PackageNotFoundError:
        missing.append(package_name)  # not installed - add it to the install list

to_install = list(missing)  # start the final install list as a copy of the missing packages
if not pkg_resources_ok:
    # Pin below version 81 specifically - anything 82+ has pkg_resources removed.
    to_install.append("setuptools<81")  # force-fix a broken/missing pkg_resources

if to_install:
    print(f"Installing/fixing packages: {to_install} (this may take a few minutes the first time)")  # tell the user what's about to happen
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + to_install)  # run pip install using this exact Python environment
    print("Done. If pkg_resources was just fixed, restart the kernel before running the next cell.")  # remind the user a restart may be needed
else:
    print("All required packages are already installed and working - nothing to do.")  # nothing needed to happen this run


In [ ]:
import base64  # encodes image bytes as text, used later to embed images directly in HTML
import io  # lets us treat in-memory bytes like a file (no disk writes needed)
import threading  # runs the dashboard's web server without blocking the rest of the notebook

import numpy as np                          # array math
import matplotlib.pyplot as plt             # plotting images and training curves
import tensorflow as tf                     # core deep learning framework - builds and trains our CNN
import tensorflow_hub as hub                # loads the pretrained locator in one line
from PIL import Image, ImageDraw, ImageFont  # drawing bounding boxes on top of images
from IPython.display import IFrame, display  # embeds the dashboard inside this notebook

print("TensorFlow version:", tf.__version__)  # confirm which TensorFlow version is active
print("GPU available:", tf.config.list_physical_devices('GPU'))  # confirm whether training will use a GPU

# Detect whether we are running in Google Colab, since the final dashboard
# cell needs to display itself slightly differently there (see Section 8).
try:
    import google.colab  # only importable inside an actual Colab environment
    IN_COLAB = True  # import succeeded - we are in Colab
except ImportError:
    IN_COLAB = False  # import failed - we are in VS Code, Jupyter, or similar

print("Running in Colab:", IN_COLAB)  # confirm which display method Section 8 will use


## Section 2 — Visualization Utilities

Reusable across the whole notebook: plotting sample data, drawing the final box, and rendering images inside the dashboard at the end all use these same functions.
---
### 🎤 Speaking Notes
- "We're building our drawing tools *before* we've touched any real data — that's deliberate. These functions know nothing about cats or dogs, they just draw whatever box you hand them."
- "Point out the three layers: `draw_bounding_box_on_image` draws ONE box, `draw_bounding_boxes_on_image` loops over MANY boxes, and `draw_bounding_boxes_on_image_array` is the one we'll actually call everywhere else, since it works directly with plain numpy arrays."
- "Say explicitly: these exact same functions get reused all the way at the end, inside the dashboard. Nothing new gets written there for drawing boxes."


In [ ]:
def draw_bounding_box_on_image(image, ymin, xmin, ymax, xmax,
                                color="lime", thickness=3,
                                display_str=None,
                                use_normalized_coordinates=True):
    """Draws ONE bounding box (with an optional text label) on a PIL image, in place.

    image                     : a PIL Image object (modified directly)
    ymin, xmin, ymax, xmax    : box coordinates, normalized to [0, 1] by default
    color                     : outline color of the box
    thickness                 : how many pixels wide the outline is
    display_str               : optional label to draw above the box, e.g. "dog: 0.93"
    use_normalized_coordinates: if True, coordinates are fractions of the image size
    """
    draw = ImageDraw.Draw(image)  # a "pen" object that can draw directly onto this image
    img_w, img_h = image.size  # actual pixel width/height of the image we're drawing on

    if use_normalized_coordinates:
        left, right, top, bottom = xmin * img_w, xmax * img_w, ymin * img_h, ymax * img_h  # convert 0-1 fractions to pixel positions
    else:
        left, right, top, bottom = xmin, xmax, ymin, ymax  # coordinates are already in pixels

    # A rectangle is 5 connected points: the 4 corners, ending back where we started.
    draw.line(
        [(left, top), (left, bottom), (right, bottom), (right, top), (left, top)],
        width=thickness, fill=color
    )  # draws the box outline as a closed 5-point line

    if display_str:
        try:
            font = ImageFont.truetype("arial.ttf", 20)  # prefer a nicer, larger system font if available
        except IOError:
            font = ImageFont.load_default()  # fall back to PIL's built-in font if Arial isn't installed

        bbox = draw.textbbox((0, 0), display_str, font=font)  # measure how big the label text will be
        text_w, text_h = bbox[2] - bbox[0], bbox[3] - bbox[1]  # extract width/height from the measured box
        margin = int(np.ceil(0.15 * text_h))  # small padding around the text, scaled to text size

        draw.rectangle(  # solid background tag so text stays readable over any photo
            [(left, top - text_h - 2 * margin), (left + text_w + 2 * margin, top)],
            fill=color
        )
        draw.text((left + margin, top - text_h - margin), display_str, fill="black", font=font)  # draw the label text on top of the background tag


def draw_box_on_image_array(image_np, ymin, xmin, ymax, xmax, display_str=None, color="lime"):
    """Numpy array in, numpy array out (with one box drawn on it)."""
    image_pil = Image.fromarray(np.uint8(image_np)).convert("RGB")  # convert the numpy array into a drawable PIL image
    draw_bounding_box_on_image(image_pil, ymin, xmin, ymax, xmax, color=color, display_str=display_str)  # draw the box in place
    return np.array(image_pil)  # convert back to numpy so it can be plotted/embedded like any other image


## Section 3 — Data Extraction and Preprocessing

Real data extraction: downloading actual cat and dog photos and preprocessing them into the fixed size, normalized format our CNN needs.

**A note on where this data comes from — the short version of a longer story:**
- `tensorflow_datasets`'s `cats_vs_dogs` relies on an original Microsoft-hosted zip with a known, unresolved bug on Windows ([tensorflow/tensorflow#84104](https://github.com/tensorflow/tensorflow/issues/84104)).
- The Google-hosted `cats_and_dogs_filtered.zip` used in TensorFlow's own tutorials has since started returning `403 Forbidden` for direct downloads.
- So instead, we pull the data from **[Hugging Face Hub](https://huggingface.co/datasets/microsoft/cats_vs_dogs)** — a dataset CDN maintained by a large infrastructure team, not a single static file. We **stream** it, so we only download the roughly 2,500 images we actually use for this class, not the full ~23,000 image dataset.

If `huggingface.co` is blocked on your network for any reason, the cell below prints a manual fallback: how to get the same dataset from Kaggle instead.
---
### 🎤 Speaking Notes
- "This is the real data extraction step. Worth being upfront with the class: getting a reliable cat/dog dataset took some trial and error — a couple of common hosting sources had download issues, so we're pulling from Hugging Face Hub, infrastructure built specifically for serving datasets reliably."
- "The key word is *streaming* — we do not download the full ~23,000 image dataset. We only pull the roughly 2,500 images we're actually going to use. Say that number out loud so it lands."
- "Walk through `examples_to_numpy` line by line as it runs: each Hugging Face example gives us a PIL image and a label — we resize, normalize to 0-1, and stack everything into plain numpy arrays before handing it to `tf.data.Dataset`."
- "If this cell fails for anyone because their network blocks Hugging Face, point them at the Kaggle fallback instructions printed in the except block — don't troubleshoot live, just note it and keep moving."


In [ ]:
IMG_SIZE = 128  # every photo gets resized to 128 x 128 before going into the CNN
N_TRAIN = 2000  # total number of training images to collect (split evenly between classes)
N_VAL = 500  # total number of validation images to collect (split evenly between classes)

SPECIES_NAMES = {0: "cat", 1: "dog"}  # matches microsoft/cats_vs_dogs' own label mapping


def load_from_huggingface():
    """Streams the dataset from Hugging Face Hub and collects a BALANCED
    number of cats and dogs.

    Important: this dataset's rows are ordered by class (all cats first,
    then all dogs) - a plain .take(N) would silently grab almost entirely
    one class, which trains a model that just learns to always guess that
    class (this is exactly what caused a suspiciously perfect near-zero
    loss the very first epoch - not a good sign, a broken-data sign).

    Instead, we walk the stream ourselves and keep collecting until we have
    an equal number of each class, then stop - still only downloading what
    we need, never the full ~23,000 image dataset.
    """
    from datasets import load_dataset  # imported here so the rest of the notebook works even if this call fails

    stream = load_dataset("microsoft/cats_vs_dogs", split="train", streaming=True)  # opens a lazy, streaming connection - nothing downloaded yet

    target_per_class_train = N_TRAIN // 2  # how many cats (and how many dogs) we need for training
    target_per_class_val = N_VAL // 2  # how many cats (and how many dogs) we need for validation

    per_class_train = {0: [], 1: []}  # buckets to collect training examples into, keyed by class
    per_class_val = {0: [], 1: []}  # buckets to collect validation examples into, keyed by class

    for example in stream:  # pulls one example at a time from the network as needed
        label = example["labels"]  # 0 = cat, 1 = dog
        if len(per_class_train[label]) < target_per_class_train:
            per_class_train[label].append(example)  # still need more of this class for training - keep it
        elif len(per_class_val[label]) < target_per_class_val:
            per_class_val[label].append(example)  # training quota met - use it for validation instead

        train_full = all(len(v) >= target_per_class_train for v in per_class_train.values())  # True once both classes have enough training examples
        val_full = all(len(v) >= target_per_class_val for v in per_class_val.values())  # True once both classes have enough validation examples
        if train_full and val_full:
            break  # stop as soon as we have enough of both classes for both splits

    train_examples = per_class_train[0] + per_class_train[1]  # combine cats and dogs into one training list
    val_examples = per_class_val[0] + per_class_val[1]  # combine cats and dogs into one validation list

    # The two classes are still in two solid blocks at this point (all
    # collected cats, then all collected dogs) - shuffle the small, already
    # in-memory lists so training sees them properly mixed.
    import random  # only needed here, for shuffling these two small lists
    random.Random(42).shuffle(train_examples)  # shuffle in place with a fixed seed for reproducibility
    random.Random(42).shuffle(val_examples)  # shuffle in place with a fixed seed for reproducibility

    return train_examples, val_examples  # both are now balanced and well-mixed


try:
    print("Streaming dataset from Hugging Face Hub (only downloads what we use)...")  # let the user know a network call is starting
    train_examples, val_examples = load_from_huggingface()  # do the actual streaming + balancing work
    print(f"Got {len(train_examples)} training and {len(val_examples)} validation examples.")  # confirm how much data we ended up with

    # Verification step: confirm the classes actually came out balanced,
    # so a data problem shows up here instead of as a confusing flat loss
    # curve several cells later.
    import collections  # provides Counter, an easy way to tally class counts
    train_counts = collections.Counter(ex["labels"] for ex in train_examples)  # count how many 0s and 1s are in the training set
    val_counts = collections.Counter(ex["labels"] for ex in val_examples)  # count how many 0s and 1s are in the validation set
    print("Training class balance:", {SPECIES_NAMES[k]: v for k, v in train_counts.items()})  # print counts using readable class names
    print("Validation class balance:", {SPECIES_NAMES[k]: v for k, v in val_counts.items()})  # print counts using readable class names
except Exception as e:
    print(f"Automatic download failed: {e}")  # show the actual error so it can be diagnosed
    print()
    print("MANUAL FALLBACK: download the dataset from Kaggle instead:")
    print("  1. Go to https://www.kaggle.com/datasets/salader/dogs-vs-cats and download it")
    print("     (a free Kaggle account is required).")
    print("  2. Extract it, then point `train_dir` / `validation_dir` at the extracted")
    print("     train/cats, train/dogs, test/cats, test/dogs folders instead of using")
    print("     the Hugging Face path below, and use tf.keras.utils.image_dataset_from_directory")
    print("     the same way earlier versions of this notebook did.")
    raise  # stop execution here rather than silently continuing with no data


In [ ]:
BATCH_SIZE = 32  # number of images processed together in one training step


def examples_to_numpy(examples, img_size):
    """Converts a list of Hugging Face examples ({'image': PIL.Image, 'labels': 0/1})
    into plain numpy arrays: resized, normalized images and float labels."""
    images = []  # will hold one resized/normalized numpy image per example
    labels = []  # will hold one float label (0.0 or 1.0) per example
    for example in examples:  # process each example one at a time
        img = example["image"].convert("RGB").resize((img_size, img_size))  # force 3 color channels and a fixed size
        images.append(np.array(img, dtype=np.float32) / 255.0)  # normalize to [0, 1]
        labels.append(float(example["labels"]))  # cast the integer label to float, matching our sigmoid output
    return np.array(images), np.array(labels)  # stack the lists into single numpy arrays


train_images, train_labels = examples_to_numpy(train_examples, IMG_SIZE)  # convert all training examples at once
val_images, val_labels = examples_to_numpy(val_examples, IMG_SIZE)  # convert all validation examples at once

print("Train images shape:", train_images.shape)  # confirm the training array's dimensions
print("Validation images shape:", val_images.shape)  # confirm the validation array's dimensions

# Wrap the numpy arrays in a tf.data.Dataset - same shuffle/batch/prefetch
# pattern used throughout this notebook, just built from arrays instead of files.
train_dataset = (
    tf.data.Dataset.from_tensor_slices((train_images, train_labels))  # turn the numpy arrays into a tf.data.Dataset
    .shuffle(1000, seed=42)  # randomize the order so the model doesn't see a fixed sequence every epoch
    .batch(BATCH_SIZE)  # group examples into batches for efficient training
    .prefetch(tf.data.AUTOTUNE)  # prepare the next batch while the current one is being processed
)
val_dataset = (
    tf.data.Dataset.from_tensor_slices((val_images, val_labels))  # turn the numpy arrays into a tf.data.Dataset
    .batch(BATCH_SIZE)  # group examples into batches (no shuffling needed for validation)
    .prefetch(tf.data.AUTOTUNE)  # prepare the next batch while the current one is being processed
)

print("Pipelines ready.")  # confirm both datasets are built and ready for training


### Visualize the extracted data

Always look at real examples before building a model on them.
---
### 🎤 Speaking Notes
- "Before building any model, always look at your data. This is a professional habit, not busywork — a quick plot like this catches label mix-ups or orientation issues immediately, before you waste time training on broken data."
- "Point at the labels under the images and confirm out loud: 0 is cat, 1 is dog, matching `SPECIES_NAMES` from the previous cell."


In [ ]:
# Pull one batch out just to plot it - .take(1) does not consume the dataset,
# it just peeks at the first batch.
for images, labels in train_dataset.take(1):  # grab exactly one batch of (images, labels)
    fig, axes = plt.subplots(1, 8, figsize=(18, 3))  # create a row of 8 side-by-side plots
    for ax, img, label in zip(axes, images[:8], labels[:8]):  # pair each subplot with one image and its label
        ax.imshow(img.numpy())  # display the image (convert from tensor to numpy first)
        ax.set_title(SPECIES_NAMES[int(label.numpy())])  # show "cat" or "dog" as the subplot title
        ax.axis("off")  # hide the x/y axis ticks, since they add no useful information here
    plt.tight_layout()  # avoid overlapping titles/images
    plt.show()  # render the figure


## Section 4 — Building Our Own CNN

A straightforward CNN: three convolution + pooling blocks to extract features at shrinking spatial resolution, then dense layers to make the final cat/dog decision.

- **Conv2D + MaxPooling2D blocks**: each block learns patterns (edges, textures, shapes) and shrinks the image, trading spatial detail for richer features - the same idea used in every CNN, including the pretrained detector we load later.
- **Dropout**: randomly turns off some neurons during training, which helps prevent the model from just memorizing the training photos.
- **A single sigmoid output**: since this is a two-class problem (cat vs dog), one output between 0 and 1 is enough - close to 0 means "cat", close to 1 means "dog".
---
### 🎤 Speaking Notes
- "Now we design the network. Walk through each block as you scroll: three Conv2D + MaxPooling pairs, each one shrinking the image spatially while increasing the number of filters — say out loud 'we're trading spatial detail for richer features' as you point at the numbers changing (32, 64, 128)."
- "Call out Dropout specifically: it's only active during training, not prediction, and it's a defense against the model just memorizing the training photos instead of learning general patterns."
- "The last layer is a single sigmoid neuron — explain that's enough for a two-class decision: close to 0 means cat, close to 1 means dog. No need for 2 output neurons here."
- "Run `model.summary()` and actually read the parameter count out loud — it builds intuition for how model size scales with image size and filter count."


In [ ]:
def build_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 3)):
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),  # declares the expected input shape: 128x128 RGB images

        tf.keras.layers.Conv2D(32, 3, activation="relu"),  # first convolution block: 32 filters, 3x3 kernel
        tf.keras.layers.MaxPooling2D(),  # halves the spatial size, keeping the strongest features

        tf.keras.layers.Conv2D(64, 3, activation="relu"),  # second convolution block: more filters, deeper features
        tf.keras.layers.MaxPooling2D(),  # halves the spatial size again

        tf.keras.layers.Conv2D(128, 3, activation="relu"),  # third convolution block: even more filters
        tf.keras.layers.MaxPooling2D(),  # halves the spatial size a third time

        tf.keras.layers.Flatten(),  # turns the 3D feature maps into a single 1D vector
        tf.keras.layers.Dense(128, activation="relu"),  # fully connected layer that combines all the extracted features
        tf.keras.layers.Dropout(0.3),  # only active during training, ignored during prediction
        tf.keras.layers.Dense(1, activation="sigmoid"),  # single output: probability of "dog"
    ])

    model.compile(
        optimizer="adam",  # a reliable, widely used default optimizer
        loss="binary_crossentropy",  # standard loss for a two-class (0/1) problem
        metrics=["accuracy"],  # track classification accuracy during training and validation
    )
    return model  # hand back the fully built and compiled model


cnn_model = build_cnn()  # create the model using the default 128x128x3 input shape
cnn_model.summary()  # print every layer, its output shape, and its parameter count


## Section 5 — Training and Evaluating Our CNN

We stream 2,000 training images and 500 validation images (see Section 3), so no further slicing is needed here. For the **live class demo**, we still train for only a handful of epochs so it finishes in a few minutes - this will not reach top accuracy, that is expected. For a properly trained model, increase `EPOCHS` and `N_TRAIN`/`N_VAL` after class, and consider adding data augmentation (random flips/rotations) to help the CNN generalize better from a relatively small dataset.
---
### 🎤 Speaking Notes
- "Set expectations before hitting run: with only 5 epochs on 2,000 images, this will NOT be highly accurate. Say that explicitly so nobody thinks something is broken when the numbers look mediocre."
- "While it trains, narrate what's happening: each epoch, the model sees every training image once, adjusts its weights based on the loss, then checks itself against validation images it never trained on."
- "Point at the printed accuracy/loss numbers as they scroll by and ask the class to guess out loud whether it's improving epoch to epoch."


In [ ]:
EPOCHS = 5  # increase this (and use the full dataset) after class for a properly trained model

history = cnn_model.fit(
    train_dataset,  # batches of training images and labels
    validation_data=val_dataset,  # batches used to check performance after each epoch (not trained on)
    epochs=EPOCHS,  # how many full passes through the training data to run
)  # history records loss/accuracy for every epoch, used for plotting later


In [ ]:
def plot_metrics(metric_name, title, ylim=1.0):
    plt.title(title)  # set the chart's title
    plt.ylim(0, ylim)  # fix the y-axis range so multiple charts are easy to compare
    plt.plot(history.history[metric_name], color="blue", label=metric_name)  # training curve
    plt.plot(history.history["val_" + metric_name], color="green", label="val_" + metric_name)  # validation curve
    plt.legend()  # show which color corresponds to which line
    plt.show()  # render the chart


plot_metrics("loss", "Training Loss")  # plot how the loss changed over training
plot_metrics("accuracy", "Training Accuracy")  # plot how accuracy changed over training


### Visualize predictions on validation photos
---
### 🎤 Speaking Notes
- "This is the moment of truth — real predictions after training. Point out the color coding: a red title means the model got that one wrong."
- "Ask the class: does the model seem to favor one class over the other? This is a good moment to casually introduce the idea of bias or class imbalance without a full lecture on it."


In [ ]:
for images, labels in val_dataset.take(1):  # grab exactly one batch of validation images and labels
    predictions = cnn_model.predict(images, verbose=0).flatten()  # values between 0 (cat) and 1 (dog)
    predicted_labels = (predictions >= 0.5).astype(int)  # convert probabilities into hard 0/1 class predictions

    fig, axes = plt.subplots(1, 8, figsize=(18, 3))  # create a row of 8 side-by-side plots
    for ax, img, true_label, pred_label, pred_score in zip(
        axes, images[:8], labels[:8].numpy().astype(int), predicted_labels[:8], predictions[:8]
    ):  # pair each subplot with its image, true label, predicted label, and confidence score
        ax.imshow(img.numpy())  # display the image (convert from tensor to numpy first)
        ax.axis("off")  # hide the x/y axis ticks
        title = f"{SPECIES_NAMES[pred_label]} ({pred_score:.2f})"  # e.g. "dog (0.87)"
        ax.set_title(title, color="black" if pred_label == true_label else "red")  # red title flags a wrong prediction
    plt.tight_layout()  # avoid overlapping titles/images
    plt.show()  # render the figure


## Section 6 — Loading a Pretrained Detector, Used Only to Find the Box

This is **SSD MobileNet V2**, trained on COCO. We use it for exactly one job: finding *where* an animal-shaped object is in a photo. We deliberately ignore whatever class label it guesses (cat, dog, or otherwise) - our own CNN from Section 4 makes that call instead.
---
### 🎤 Speaking Notes
- "Now we bring in a second model, and it does something completely different from our CNN — say that distinction clearly."
- "This locator has never seen our training data, and we are going to deliberately ignore its own cat/dog guess entirely — that's the whole design of this project."
- "Explain `COCO_ANIMAL_CLASSES` in plain terms: we're only using it to answer 'is there an animal-shaped thing here,' never 'which animal is it.'"
- "Good spot to contrast 'transfer learning by inference' (this cell) against 'training your own model' (Sections 4-5) — ask the class which one took longer to build and which one took longer to run."


In [ ]:
LOCATOR_URL = "https://tfhub.dev/tensorflow/ssd_mobilenet_v2/2"  # the pretrained model's address on TensorFlow Hub

print("Loading pretrained locator... (first run may take a minute or two)")  # first run downloads and caches the model
locator = hub.load(LOCATOR_URL)  # download (or load from cache) and prepare the model for use
print("Locator loaded successfully")  # confirm the model is ready

# COCO class ids for cat and dog - we use these only to decide "is there an
# animal-shaped thing here worth cropping", not to decide which one it is.
COCO_ANIMAL_CLASSES = {16, 17, 18, 19, 20, 21, 22, 23, 24, 25}  # bird, cat, dog, horse, sheep, cow, elephant, bear, zebra, giraffe
LOCATOR_THRESHOLD = 0.4  # minimum confidence required before we trust a detected box


def find_animal_box(image_np):
    """Runs the pretrained locator and returns the single most confident
    animal-shaped box, or None if nothing confident enough was found."""
    input_tensor = tf.convert_to_tensor(image_np)[tf.newaxis, ...]  # convert to a tensor and add the batch dimension the model expects

    detections = locator(input_tensor)  # run the pretrained model on this one image

    boxes = detections["detection_boxes"][0].numpy()               # (ymin, xmin, ymax, xmax), normalized
    classes = detections["detection_classes"][0].numpy().astype(int)  # COCO class id for each detected box
    scores = detections["detection_scores"][0].numpy()  # confidence score for each detected box

    keep = (scores >= LOCATOR_THRESHOLD) & np.isin(classes, list(COCO_ANIMAL_CLASSES))  # True for confident, animal-shaped detections only
    if not np.any(keep):
        return None  # no confident animal-shaped region found

    # Take the single most confident box among the candidates.
    best_index = np.argmax(scores * keep)  # picks the highest-scoring detection among the kept ones
    ymin, xmin, ymax, xmax = boxes[best_index]  # unpack that box's four coordinates
    return float(ymin), float(xmin), float(ymax), float(xmax)  # return as plain Python floats


## Section 7 — Combining Both: Detect the Box, Classify with Our Own CNN

The full pipeline for one photo:
1. Run the **pretrained locator** to find the animal's box (Section 6).
2. **Crop** the image to that box.
3. Resize and normalize the crop the same way we did for training (Section 3).
4. Run **our own trained CNN** (Section 4-5) on the crop to decide cat or dog.
5. Draw the box with **our CNN's** label and confidence - not the locator's.

If the locator finds nothing confident enough, we fall back to classifying the whole photo instead of failing outright.
---
### 🎤 Speaking Notes
- "This is where the two models actually meet. Walk through `detect_and_classify` step by step as it's on screen: locate the box, crop to it, resize the crop, then classify with our own model."
- "Emphasize the fallback path out loud: if the locator finds nothing confident, we still classify the whole photo instead of failing outright. Ask the class: why is that graceful fallback better than just crashing in a real product?"
- "When the test cell runs, narrate the four sample photos as they render — call out true vs. predicted for each one before moving on."


In [ ]:
def detect_and_classify(image_np):
    """Runs the full pipeline on one image and returns the annotated image
    plus a plain description of what was found."""
    img_h, img_w = image_np.shape[0], image_np.shape[1]  # actual pixel dimensions of the input photo
    box = find_animal_box(image_np)  # ask the pretrained locator where the animal is (or None)

    if box is not None:
        ymin, xmin, ymax, xmax = box  # unpack the normalized (0-1) box coordinates
        # Convert normalized coordinates to actual pixel coordinates to crop with.
        top, bottom = int(ymin * img_h), int(ymax * img_h)  # vertical crop boundaries in pixels
        left, right = int(xmin * img_w), int(xmax * img_w)  # horizontal crop boundaries in pixels
        crop = image_np[top:bottom, left:right]  # slice out just the located region
    else:
        crop = image_np  # fall back to the whole photo if nothing confident was located

    # Preprocess the crop exactly like training data: resize to IMG_SIZE, scale to [0, 1].
    crop_resized = tf.image.resize(crop, (IMG_SIZE, IMG_SIZE))  # match the size our CNN was trained on
    crop_resized = tf.cast(crop_resized, tf.float32) / 255.0  # normalize pixel values to [0, 1]
    crop_batch = tf.expand_dims(crop_resized, axis=0)  # add the batch dimension the model expects

    score = float(cnn_model.predict(crop_batch, verbose=0)[0][0])  # our own CNN's raw prediction: 0 (cat) to 1 (dog)
    label = SPECIES_NAMES[int(score >= 0.5)]  # convert the score into a "cat"/"dog" label
    confidence = score if label == "dog" else 1 - score  # confidence in whichever label was chosen

    display_str = f"{label}: {confidence:.2f}"  # text to draw on the image, e.g. "dog: 0.87"

    if box is not None:
        annotated = draw_box_on_image_array(image_np, ymin, xmin, ymax, xmax, display_str=display_str)  # draw the box and label on the original photo
    else:
        annotated = np.array(Image.fromarray(np.uint8(image_np)))  # no box to draw

    detection = {"label": label, "confidence": round(confidence, 3), "box_found": box is not None}  # plain summary of the result
    return annotated, detection  # hand back both the annotated image and the result details


In [ ]:
# Grab a few real raw photos directly from the validation examples we already
# streamed, to test the full pipeline end-to-end - the same way an uploaded
# photo would arrive in the dashboard (a plain full-size image, not a
# preprocessed/resized dataset batch).
sample_examples = [ex for ex in val_examples if ex["labels"] == 0][:2] + \
                   [ex for ex in val_examples if ex["labels"] == 1][:2]  # 2 cats + 2 dogs

fig, axes = plt.subplots(1, len(sample_examples), figsize=(16, 4))  # one subplot per sample photo
for ax, example in zip(axes, sample_examples):  # pair each subplot with one sample example
    image = np.array(example["image"].convert("RGB"))  # convert the PIL image to a plain numpy array
    annotated, detection = detect_and_classify(image)  # run the full locate + classify pipeline
    true_label = SPECIES_NAMES[example["labels"]]  # the actual correct label for this photo
    ax.imshow(annotated)  # display the annotated (boxed and labeled) result
    ax.set_title(f"true: {true_label} | predicted: {detection['label']} ({detection['confidence']})")  # show both true and predicted labels
    ax.axis("off")  # hide the x/y axis ticks
plt.tight_layout()  # avoid overlapping titles/images
plt.show()  # render the figure


## Section 8 — The Dashboard

Everything from Sections 6-7 (`find_animal_box`, `detect_and_classify`) gets reused directly here - the dashboard adds nothing new on the detection or classification side, only a small web page around it.

**How it works:**
1. A small **Flask** web server is started in a background thread, so it runs without blocking the rest of the notebook.
2. The page lets you upload a photo through a simple form.
3. On upload, the server runs `detect_and_classify` - the pretrained locator finds the box, **our own trained CNN** decides cat or dog.
4. The annotated image is encoded directly as a base64 string and embedded in the page - no files are saved to disk.
5. The dashboard is then displayed **right here, inline, in the notebook output**.
---
### 🎤 Speaking Notes
- "This is the payoff — everything becomes a tool people can actually use. But make sure to say explicitly: we are NOT writing any new detection logic here, we're just wrapping Sections 6 and 7 in a web page."
- "Briefly explain `image_to_data_uri`: we never save a file to disk anywhere in this dashboard, we just encode the result as text and embed it directly into the HTML response."
- "When you reach the threading cell, explain it in plain terms: normally a web server blocks your program forever once started. Running it in a background thread lets the rest of the notebook — and the IFrame display right after it — keep working alongside it."
- "Have 2-3 test photos ready on your desktop *before* class starts, so you can actually upload one live the moment the dashboard renders, instead of hunting for an image on the spot."


In [ ]:
def image_to_data_uri(image_np):
    """Encodes a numpy image as a base64 PNG data URI, so it can be embedded
    directly inside an HTML <img> tag with no file ever being saved to disk."""
    image_pil = Image.fromarray(np.uint8(image_np))  # convert the numpy array into a PIL image
    buffer = io.BytesIO()  # an in-memory "file" to hold the encoded image bytes
    image_pil.save(buffer, format="PNG")  # write the image into that in-memory buffer as PNG
    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")  # turn the raw bytes into a text string
    return f"data:image/png;base64,{encoded}"  # a data URI a browser can display directly, no file needed


In [ ]:
from flask import Flask, render_template_string, request  # the web framework powering the dashboard

flask_app = Flask(__name__)  # create the web application object
ALLOWED_EXTENSIONS = {"png", "jpg", "jpeg", "webp"}  # file types the upload form will accept


def allowed_file(filename):
    return "." in filename and filename.rsplit(".", 1)[1].lower() in ALLOWED_EXTENSIONS  # True only if the extension is in our allowed set


PAGE_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Cat &amp; Dog Detection Dashboard</title>
    <style>
        :root {
            --accent: #16a34a; --accent-dark: #15803d; --bg: #f6f7f9;
            --card-bg: #ffffff; --text: #1f2937; --muted: #6b7280; --border: #e5e7eb;
        }
        * { box-sizing: border-box; }
        body { margin: 0; font-family: -apple-system, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
               background: var(--bg); color: var(--text); }
        .topbar { text-align: center; padding: 24px 16px 16px; }
        .topbar h1 { margin: 4px 0; font-size: 24px; }
        .subtitle { color: var(--muted); margin: 0; font-size: 14px; }
        main { max-width: 560px; margin: 0 auto; padding: 0 16px 32px; }
        .upload-card, .result-card {
            background: var(--card-bg); border: 1px solid var(--border); border-radius: 12px;
            padding: 20px; margin-top: 16px; text-align: center;
        }
        .file-label {
            display: block; border: 2px dashed var(--border); border-radius: 10px;
            padding: 24px 16px; cursor: pointer; color: var(--muted);
        }
        .file-label:hover { border-color: var(--accent); }
        .file-label input[type="file"] { display: none; }
        button {
            margin-top: 14px; background: var(--accent); color: white; border: none;
            padding: 9px 24px; border-radius: 8px; font-size: 15px; cursor: pointer;
        }
        button:hover { background: var(--accent-dark); }
        .notice { color: #b45309; margin-top: 12px; font-size: 13px; }
        .result-card h2 { margin-top: 0; font-size: 18px; }
        .result-image { max-width: 100%; border-radius: 10px; border: 1px solid var(--border); }
        .detections-list { list-style: none; padding: 0; margin-top: 14px; text-align: left; }
        .detections-list li { padding: 6px 0; border-bottom: 1px solid var(--border); font-size: 14px; }
        .detections-list li:last-child { border-bottom: none; }
    </style>
</head>
<body>
    <header class="topbar">
        <h1>Cat &amp; Dog Detection</h1>
        <p class="subtitle">Upload a photo — a pretrained model finds the animal, our own trained CNN decides cat or dog</p>
    </header>
    <main>
        <section class="upload-card">
            <form action="/predict" method="post" enctype="multipart/form-data">
                <label for="photo" class="file-label">
                    <span id="file-chosen">Click to choose an image</span>
                    <input type="file" id="photo" name="photo" accept=".png,.jpg,.jpeg,.webp" required>
                </label>
                <button type="submit">Detect</button>
            </form>
            {% if error %}<p class="notice">{{ error }}</p>{% endif %}
        </section>
        {% if result_image %}
        <section class="result-card">
            <h2>Result</h2>
            <img src="{{ result_image }}" alt="Detection result" class="result-image">
            {% if detection %}
            <ul class="detections-list">
                <li><strong>{{ detection.label|capitalize }}</strong> — confidence {{ detection.confidence }}</li>
                <li>{{ "Animal region located by the pretrained model" if detection.box_found else "No confident region found - classified the whole photo" }}</li>
            </ul>
            {% endif %}
        </section>
        {% endif %}
    </main>
    <script>
        const input = document.getElementById('photo');
        const label = document.getElementById('file-chosen');
        input.addEventListener('change', () => {
            if (input.files.length > 0) { label.textContent = input.files[0].name; }
        });
    </script>
</body>
</html>
"""


@flask_app.route("/", methods=["GET"])  # the homepage - shown when someone first opens the dashboard
def index():
    return render_template_string(PAGE_TEMPLATE, result_image=None, detection=None, error=None)  # render the empty upload form


@flask_app.route("/predict", methods=["POST"])  # handles the form submission when a photo is uploaded
def predict():
    if "photo" not in request.files or request.files["photo"].filename == "":
        return render_template_string(PAGE_TEMPLATE, result_image=None, detection=None,
                                       error="Please choose an image file first.")  # no file was actually selected

    file = request.files["photo"]  # the uploaded file object
    if not allowed_file(file.filename):
        return render_template_string(PAGE_TEMPLATE, result_image=None, detection=None,
                                       error="Please upload a PNG, JPG, JPEG, or WEBP image.")  # reject unsupported file types

    # Read the uploaded file straight from memory - no disk write needed.
    image = Image.open(file.stream).convert("RGB")  # open the upload directly from memory
    image_np = np.array(image)  # convert to a numpy array for our pipeline functions

    # Reuses the exact same pipeline from Section 7 above.
    annotated_np, detection = detect_and_classify(image_np)  # locate the animal and classify it
    result_image = image_to_data_uri(annotated_np)  # encode the annotated result for embedding in HTML

    return render_template_string(PAGE_TEMPLATE, result_image=result_image, detection=detection, error=None)  # render the page with the result

print("Flask app defined.")  # confirm the app and its routes are ready, before we start the server


### Launching the dashboard, right here in the notebook


In [ ]:
def run_flask_app():
    # use_reloader=False is required when running Flask inside a background
    # thread - the reloader tries to restart the whole process, which does
    # not work once we are already inside a thread instead of the main process.
    flask_app.run(host="127.0.0.1", port=5000, debug=False, use_reloader=False)  # start the server, blocking this thread only


dashboard_thread = threading.Thread(target=run_flask_app, daemon=True)  # prepare a background thread to run the server
dashboard_thread.start()  # start the server without blocking the rest of the notebook

print("Dashboard starting...")  # give the server a moment to come up before we try to display it

if IN_COLAB:
    from google.colab import output as colab_output  # only available inside Colab
    colab_output.serve_kernel_port_as_iframe(5000, height=650)  # Colab-specific way to proxy and display the local port
else:
    display(IFrame(src="http://127.0.0.1:5000", width="100%", height=650))  # embed the dashboard directly in the notebook output


**Upload a photo of a cat or dog in the dashboard above.** The green box comes from the pretrained locator; the label and confidence next to it come from **your own trained CNN**.

If the dashboard area above looks blank, re-run the cell above once (the server sometimes needs a moment to finish starting before the page loads).


## Wrap Up and Next Steps

**What we built, start to finish, all in this one notebook:**
- A real data pipeline: extracting and preprocessing actual cat/dog photos
- Our **own CNN**, built and trained from scratch, that decides cat vs dog
- A pretrained locator, used only to find where the animal is - never to classify it
- A combined pipeline: locate with the pretrained model, classify with our own model
- A live dashboard, running in a background thread and displayed inline, using that exact combined pipeline

**Where to go from here:**
- Increase `EPOCHS`, `N_TRAIN`, and `N_VAL` for a much more accurate CNN.
- Add data augmentation (random flips, rotations, zooms) to help the CNN generalize better from limited training data.
- Try swapping the locator for a different TensorFlow Hub model and compare speed/accuracy of the box it finds.
---
### 🎤 Speaking Notes
- "Recap in one sentence: we built our own brain for classification, borrowed someone else's eyes for finding the animal, and wrapped both in a tool people can actually use."
- "Don't just read the 'Where to go from here' bullets — pick one and actually discuss it live, e.g. 'what do you think happens to accuracy if we bump EPOCHS to 20 and N_TRAIN to the full dataset?' and take a guess from the class before revealing why."
